# 📧 Spam Email Classifier (NLP)

A complete spam/ham (not-spam) text classifier built with **Python**, **Scikit-learn**, and basic **NLP** techniques.

**Pipeline:**
1. Load dataset (SMS Spam Collection — 5,572 real labeled messages)
2. Clean & preprocess text
3. Convert text → numeric features (TF-IDF)
4. Train models (Naive Bayes + Logistic Regression)
5. Evaluate (accuracy, precision, recall, F1, confusion matrix)
6. Try your own messages interactively

> Just click **Runtime → Run all** in Colab. No file uploads needed — the dataset downloads automatically.


## 1. Install & Import Libraries

In [ ]:
# These come pre-installed on Colab, but this ensures everything is available
!pip install scikit-learn pandas matplotlib seaborn nltk -q


In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

print("✅ Libraries imported successfully")


## 2. Load the Dataset

We use the **SMS Spam Collection Dataset** — a well-known public dataset with 5,572 messages
labeled as `spam` or `ham` (not spam). It's hosted on GitHub as a clean CSV, so we load it
directly with `pandas` — no manual upload needed.


In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Quick look at class balance
print(df['label'].value_counts())

sns.countplot(x='label', data=df, palette='Set2')
plt.title("Spam vs Ham — Class Distribution")
plt.show()


## 3. Text Preprocessing

Raw text needs cleaning before a model can learn from it. We will:
- Lowercase everything
- Remove punctuation and numbers
- Remove common English **stopwords** (e.g. "the", "is", "and") that carry little meaning
- Strip extra whitespace

This is standard NLP preprocessing and keeps only the words that actually help distinguish spam from ham.


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                                  # lowercase
    text = re.sub(r'\d+', '', text)                       # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = ' '.join(word for word in text.split() if word not in stop_words)  # remove stopwords
    text = re.sub(r'\s+', ' ', text).strip()              # remove extra whitespace
    return text

df['clean_message'] = df['message'].apply(clean_text)

# Compare before/after
df[['message', 'clean_message']].head()


In [ ]:
# Encode labels: ham -> 0, spam -> 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()


## 4. Feature Extraction — TF-IDF

Machine learning models need numbers, not raw text. We use **TF-IDF (Term Frequency–Inverse
Document Frequency)**, which converts each message into a vector of word importance scores —
words that are frequent in a message but rare across all messages get higher weight (these tend
to be the most distinguishing words).


In [ ]:
X = df['clean_message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training samples:", X_train_tfidf.shape)
print("Test samples:", X_test_tfidf.shape)


## 5. Train the Models

We train two classic, fast, and well-suited models for text classification and compare them:

- **Multinomial Naive Bayes** — a classic baseline for text/spam classification, very fast and surprisingly strong
- **Logistic Regression** — a reliable linear classifier, often slightly more accurate on TF-IDF features


In [ ]:
# Model 1: Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print("✅ Both models trained")


## 6. Evaluate the Models

In [ ]:
def evaluate(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print()
    return acc, prec, rec, f1

nb_scores = evaluate(y_test, nb_preds, "Naive Bayes")
lr_scores = evaluate(y_test, lr_preds, "Logistic Regression")


In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, name in zip(axes, [nb_preds, lr_preds], ["Naive Bayes", "Logistic Regression"]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'], ax=ax)
    ax.set_title(f"{name} — Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
print("Detailed report — Logistic Regression\n")
print(classification_report(y_test, lr_preds, target_names=['Ham', 'Spam']))


## 7. Pick the Best Model

We'll go with **Logistic Regression** for the final predictor if it performs as well or better
(this is typical on this dataset), since it tends to generalize slightly better. Feel free to
swap to `nb_model` below if you find Naive Bayes performs better for you.


In [ ]:
best_model = lr_model if lr_scores[3] >= nb_scores[3] else nb_model
print("Selected model:", type(best_model).__name__)


## 8. Try It Yourself 🔍

Type any message below and the model will predict whether it's **Spam** or **Ham**.
Edit the `sample_messages` list and re-run the cell to test your own examples.


In [ ]:
def predict_message(message, model=best_model):
    cleaned = clean_text(message)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][pred]
    label = "🚨 SPAM" if pred == 1 else "✅ HAM (Not Spam)"
    return label, prob

sample_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow at 1pm?",
    "URGENT: Your account has been suspended. Verify your identity immediately at this link.",
    "Don't forget to submit the assignment before midnight, professor said no extensions.",
    "FREE entry in a weekly competition to win an iPhone 15! Text WIN to 80085 now!"
]

for msg in sample_messages:
    label, prob = predict_message(msg)
    print(f"Message : {msg}")
    print(f"Prediction: {label}  (confidence: {prob:.2%})")
    print("-" * 80)


In [ ]:
# Try your own message here 👇
your_message = "Type your message here"

label, prob = predict_message(your_message)
print(f"Message   : {your_message}")
print(f"Prediction: {label}")
print(f"Confidence: {prob:.2%}")


## 9. (Optional) Save the Model

If you later want to reuse this model in a web app (Flask/FastAPI), you can save the trained
model and vectorizer with `joblib` and load them elsewhere.


In [ ]:
import joblib

joblib.dump(best_model, 'spam_classifier_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("✅ Saved: spam_classifier_model.pkl, tfidf_vectorizer.pkl")
print("(Find them in the Colab file browser on the left, under Files)")


## Summary

- **Dataset**: SMS Spam Collection (5,572 messages, public dataset)
- **Preprocessing**: lowercasing, punctuation/number removal, stopword removal
- **Feature extraction**: TF-IDF (top 3,000 terms)
- **Models compared**: Multinomial Naive Bayes vs Logistic Regression
- **Evaluation**: Accuracy, Precision, Recall, F1-score, Confusion Matrix
- **Result**: A reusable function `predict_message()` that classifies any new text as spam or ham

This single notebook covers the full NLP classification pipeline described in the project: *data
loading → cleaning → vectorization → model training → evaluation → live inference.*
